# US 15 — Evolução Temporal dos Rendimentos e Bens de um Agente Político

**Descrição:**  
Como membro do Comité de Ética, quero examinar como os rendimentos e o património de um determinado agente político evoluem ao longo do tempo. Esta análise produz três gráficos separados:

1. Evolução do **salário bruto** ao longo de todas as declarações.
2. Evolução dos **rendimentos suplementares**, distinguindo os diferentes tipos.
3. Evolução dos **bens declarados**, distinguindo as diferentes categorias de ativos.

São consideradas **todas** as declarações do agente: initial, regular e exceptional, ordenadas cronologicamente.

---

## Conceitos Teóricos Aplicados

### Série Temporal
Uma série temporal é um conjunto de observações $\{x_t\}$ ordenadas no tempo $t_1 < t_2 < \ldots < t_n$.  
Cada declaração de interesses constitui uma observação num instante de tempo $t_i$.

### Tendência (Trend)
A tendência representa a direção geral da série ao longo do tempo:  
- **Crescente**: o valor aumenta progressivamente.
- **Decrescente**: o valor diminui progressivamente.
- **Estável**: sem variação sistemática ao longo do tempo.

### Total de Ativos Declarados
Como o dataset não inclui passivos, calcula-se o total de ativos declarados como:
$$\text{total\_assets}_t = \sum_{c \in \text{ASSET\_COLS}} \text{valor}_c(t)$$

### Gráfico de Linha (Line Chart)
Representação adequada para séries temporais: o eixo horizontal representa o tempo e o eixo vertical o valor observado. Permite identificar tendências, ciclos e variações abruptas entre declarações.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from pathlib import Path

In [ ]:
def load_declarations_data(path: str):
    """Load and prepare the declarations dataset.

    Returns df, side_income_cols, asset_cols.
    Columns are detected automatically from column name prefixes.
    """
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    df["declaration_date"] = pd.to_datetime(df["declaration_date"])

    side_income_cols = [col for col in df.columns if col.startswith("side_income_")]
    asset_cols       = [col for col in df.columns if col.startswith("assets_in_")]

    # total_assets: sum of all declared asset categories
    df["total_assets"] = df[asset_cols].sum(axis=1)

    return df, side_income_cols, asset_cols


DATA_PATH = "../Dados/dataset1_declarations.csv"
df, SIDE_INCOME_COLS, ASSET_COLS = load_declarations_data(DATA_PATH)

print(f"Dataset carregado  : {df.shape[0]} declarações, {df['agent_id'].nunique()} agentes únicos")
print(f"Side income cols   : {SIDE_INCOME_COLS}")
print(f"Asset cols         : {ASSET_COLS}")
print(f"Tipos de declaração: {sorted(df['declaration_type'].unique())}")

## Notas Metodológicas

### Net Worth

Como o dataset das declarações não inclui passivos, dívidas ou outros componentes patrimoniais negativos (e.g., `liabilities`, `debts`, `loans`), não é possível calcular diretamente o valor líquido patrimonial (*net worth*). Assim, esta análise usa os **ativos declarados por categoria** e calcula `total_assets` como a soma das categorias de ativos disponíveis.

### Dataset 2 — Holdings

O ficheiro `dataset2_holdings.csv` contém informação detalhada ao nível empresa sobre participações acionistas (NIF da empresa, valor total em ações, percentagem da empresa). No entanto, **não é utilizado como fonte principal nesta US15** porque:
- não contém `declaration_id`, impossibilitando a ligação direta às declarações;
- as datas não correspondem diretamente às datas das declarações do dataset 1.

O valor de ações declarado utilizado nesta análise provém da coluna `assets_in_stocks` do `dataset1_declarations.csv`, que agrega o valor total de ações por declaração.

In [ ]:
# ── Funções de extração de dados ─────────────────────────────────────────────

def get_actor_declarations(df: pd.DataFrame, agent_id: str) -> pd.DataFrame:
    """Filter all declarations for a given agent, sorted by date.

    Includes all declaration types: initial, regular, exceptional.
    """
    actor_df = df[df["agent_id"] == agent_id].copy()
    if actor_df.empty:
        raise ValueError(f"Agente '{agent_id}' não encontrado no dataset.")
    return actor_df.sort_values("declaration_date").reset_index(drop=True)


def extract_gross_salary_series(actor_df: pd.DataFrame) -> pd.DataFrame:
    """Extract gross salary time series from actor declarations."""
    return actor_df[["declaration_date", "declaration_type", "gross_salary"]].copy()


def extract_side_income_series(actor_df: pd.DataFrame, side_income_cols: list) -> pd.DataFrame:
    """Melt side income columns into long format with readable type labels."""
    long_df = actor_df[["declaration_date", "declaration_type"] + side_income_cols].melt(
        id_vars=["declaration_date", "declaration_type"],
        value_vars=side_income_cols,
        var_name="side_income_type",
        value_name="amount",
    )
    long_df["side_income_type"] = (
        long_df["side_income_type"]
        .str.replace("side_income_", "", regex=False)
        .str.replace("_", " ")
        .str.title()
    )
    return long_df


def extract_assets_series(actor_df: pd.DataFrame, asset_cols: list) -> pd.DataFrame:
    """Melt asset columns into long format with readable category labels."""
    long_df = actor_df[["declaration_date", "declaration_type"] + asset_cols].melt(
        id_vars=["declaration_date", "declaration_type"],
        value_vars=asset_cols,
        var_name="asset_category",
        value_name="value",
    )
    long_df["asset_category"] = (
        long_df["asset_category"]
        .str.replace("assets_in_", "", regex=False)
        .str.replace("_", " ")
        .str.title()
    )
    return long_df


# ── Funções de visualização ───────────────────────────────────────────────────

def plot_gross_salary(gross_df: pd.DataFrame, agent_id: str) -> None:
    """Line chart of gross salary evolution over time.

    Uses distinct markers per declaration type.
    """
    marker_map = {"initial": "s", "regular": "o", "exceptional": "^"}
    color_map  = {"initial": "#2196F3", "regular": "#4CAF50", "exceptional": "#FF5722"}

    fig, ax = plt.subplots(figsize=(12, 6))

    ax.plot(
        gross_df["declaration_date"],
        gross_df["gross_salary"],
        color="#90A4AE", linewidth=1.2, linestyle="--", zorder=1,
    )

    for dtype, group in gross_df.groupby("declaration_type"):
        ax.scatter(
            group["declaration_date"],
            group["gross_salary"],
            label=dtype.capitalize(),
            marker=marker_map.get(dtype, "o"),
            color=color_map.get(dtype, "grey"),
            s=80, zorder=2,
        )

    ax.set_title(
        f"Evolution of Gross Salary — Agent {agent_id}",
        fontsize=14, fontweight="bold", pad=15,
    )
    ax.set_xlabel("Declaration date", fontsize=12)
    ax.set_ylabel("Gross salary (€)", fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
    ax.legend(title="Declaration type", frameon=True)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.grid(axis="x", linestyle=":", alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("US15_gross_salary.svg", format="svg", bbox_inches="tight")
    plt.show()
    print("Gráfico guardado em: US15_gross_salary.svg")


def plot_side_income(side_income_df: pd.DataFrame, agent_id: str) -> None:
    """Multi-line chart of side income by type over time."""
    colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    tipos  = sorted(side_income_df["side_income_type"].unique())

    fig, ax = plt.subplots(figsize=(12, 6))

    for i, tipo in enumerate(tipos):
        grupo = (
            side_income_df[side_income_df["side_income_type"] == tipo]
            .sort_values("declaration_date")
        )
        ax.plot(
            grupo["declaration_date"],
            grupo["amount"],
            marker="o", linewidth=1.8, label=tipo,
            color=colors[i % len(colors)],
        )

    ax.set_title(
        f"Evolution of Side Income by Type — Agent {agent_id}",
        fontsize=14, fontweight="bold", pad=15,
    )
    ax.set_xlabel("Declaration date", fontsize=12)
    ax.set_ylabel("Side income amount (€)", fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
    ax.legend(title="Income type", frameon=True)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.grid(axis="x", linestyle=":", alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("US15_side_income.svg", format="svg", bbox_inches="tight")
    plt.show()
    print("Gráfico guardado em: US15_side_income.svg")


def plot_assets(assets_df: pd.DataFrame, agent_id: str) -> None:
    """Multi-line chart of declared assets by category over time."""
    colors     = plt.rcParams["axes.prop_cycle"].by_key()["color"]
    categorias = sorted(assets_df["asset_category"].unique())

    fig, ax = plt.subplots(figsize=(12, 6))

    for i, categoria in enumerate(categorias):
        grupo = (
            assets_df[assets_df["asset_category"] == categoria]
            .sort_values("declaration_date")
        )
        ax.plot(
            grupo["declaration_date"],
            grupo["value"],
            marker="o", linewidth=1.8, label=categoria,
            color=colors[(i + 3) % len(colors)],
        )

    ax.set_title(
        f"Evolution of Declared Assets by Category — Agent {agent_id}",
        fontsize=14, fontweight="bold", pad=15,
    )
    ax.set_xlabel("Declaration date", fontsize=12)
    ax.set_ylabel("Asset value (€)", fontsize=12)
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f} €"))
    ax.legend(title="Asset category", frameon=True)
    ax.grid(axis="y", linestyle="--", alpha=0.5)
    ax.grid(axis="x", linestyle=":", alpha=0.3)
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig("US15_assets.svg", format="svg", bbox_inches="tight")
    plt.show()
    print("Gráfico guardado em: US15_assets.svg")


# ── Função de interpretação ───────────────────────────────────────────────────

def print_interpretation(
    actor_df: pd.DataFrame,
    gross_df: pd.DataFrame,
    side_income_df: pd.DataFrame,
    assets_df: pd.DataFrame,
    agent_id: str,
) -> None:
    """Print a descriptive interpretation of the three time series."""
    # Gross salary trend
    first_salary  = gross_df["gross_salary"].iloc[0]
    last_salary   = gross_df["gross_salary"].iloc[-1]
    salary_pct    = (last_salary - first_salary) / first_salary * 100
    salary_trend  = (
        "increasing" if salary_pct > 0
        else ("decreasing" if salary_pct < 0 else "stable")
    )

    # Side income
    mean_by_type    = side_income_df.groupby("side_income_type")["amount"].mean()
    top_income_type = mean_by_type.idxmax()

    # Assets
    mean_by_asset = assets_df.groupby("asset_category")["value"].mean()
    top_asset_cat = mean_by_asset.idxmax()

    # Total assets evolution
    first_total = actor_df["total_assets"].iloc[0]
    last_total  = actor_df["total_assets"].iloc[-1]
    assets_pct  = (last_total - first_total) / first_total * 100
    assets_trend = (
        "increasing" if assets_pct > 0
        else ("decreasing" if assets_pct < 0 else "stable")
    )

    type_counts = actor_df["declaration_type"].value_counts()

    print("=" * 70)
    print("DESCRIPTIVE INTERPRETATION")
    print("=" * 70)
    print(f"  Agent              : {agent_id}")
    print(f"  Role               : {actor_df['role'].iloc[0]}")
    print(f"  Declarations used  : {len(actor_df)} — {type_counts.to_dict()}")
    print(f"  Period             : {actor_df['declaration_date'].min().date()} "
          f"to {actor_df['declaration_date'].max().date()}")
    print()
    print(f"  Gross Salary — {salary_trend} trend ({salary_pct:+.1f}%)")
    print(f"    First declaration : {first_salary:>12,.0f} €")
    print(f"    Last declaration  : {last_salary:>12,.0f} €")
    print()
    print(f"  Side Income — most relevant type by average: {top_income_type}")
    for tipo, media in mean_by_type.sort_values(ascending=False).items():
        print(f"    {tipo:<28}: {media:>10,.0f} €")
    print()
    print(f"  Declared Assets — largest average category: {top_asset_cat}")
    for cat, media in mean_by_asset.sort_values(ascending=False).items():
        print(f"    {cat:<28}: {media:>10,.0f} €")
    print()
    print(f"  Total Assets — {assets_trend} trend ({assets_pct:+.1f}%)")
    print(f"    First declaration : {first_total:>12,.0f} €")
    print(f"    Last declaration  : {last_total:>12,.0f} €")
    print()
    print("  Note: this analysis is purely descriptive and based on declared")
    print("  values. It does not support legal conclusions about the agent.")
    print("=" * 70)


# ── Função principal ──────────────────────────────────────────────────────────

def run_us15_analysis(
    df: pd.DataFrame,
    side_income_cols: list,
    asset_cols: list,
    selected_agent_id: str,
) -> None:
    """Run the full US15 analysis for a given political agent.

    Produces three separate graphs:
      1. Gross salary evolution.
      2. Side income by type evolution.
      3. Declared assets by category evolution.
    """
    # 1 — Filter declarations
    actor_df = get_actor_declarations(df, selected_agent_id)

    type_counts = actor_df["declaration_type"].value_counts()
    print(f"Agente selecionado  : {selected_agent_id}")
    print(f"Cargo               : {actor_df['role'].iloc[0]}")
    print(f"Instituição         : {actor_df['institution'].iloc[0]}")
    print(f"Total declarações   : {len(actor_df)} — {type_counts.to_dict()}")
    print(f"Período             : {actor_df['declaration_date'].min().date()} "
          f"a {actor_df['declaration_date'].max().date()}")
    print()

    # 2 — Extract series
    gross_df       = extract_gross_salary_series(actor_df)
    side_income_df = extract_side_income_series(actor_df, side_income_cols)
    assets_df      = extract_assets_series(actor_df, asset_cols)

    # 3 — Show data tables
    cols_view = ["declaration_date", "declaration_type", "gross_salary", "total_assets"]
    display_df = actor_df[cols_view].assign(
        declaration_date=actor_df["declaration_date"].dt.strftime("%Y-%m-%d"),
        gross_salary=actor_df["gross_salary"].map("{:,.2f} €".format),
        total_assets=actor_df["total_assets"].map("{:,.2f} €".format),
    )
    print("Declarações do agente (visão geral):")
    print(display_df.to_string(index=False))
    print()

    # 4 — Graphs
    plot_gross_salary(gross_df, selected_agent_id)
    plot_side_income(side_income_df, selected_agent_id)
    plot_assets(assets_df, selected_agent_id)

    # 5 — Interpretation
    print()
    print_interpretation(actor_df, gross_df, side_income_df, assets_df, selected_agent_id)


print("Funções definidas com sucesso.")

## Análise — Selecção do Agente e Execução

Altere `selected_agent_id` para analisar um agente político diferente.  
O agente `A00002` é usado como exemplo por possuir os três tipos de declaração: initial, regular e exceptional.

In [ ]:
# ── Selecção do agente (alterar aqui) ────────────────────────────────────────
selected_agent_id = "A00002"

run_us15_analysis(df, SIDE_INCOME_COLS, ASSET_COLS, selected_agent_id)